## IMDB Sentiment Classifier with BERT

A minimal walkthrough of fine-tuning **BERT** on the IMDB movie reviews dataset for binary sentiment classification (positive / negative).

**What we'll do:**
1. Install libraries and set up
2. Load the IMDB dataset
3. Look at the data
4. Split into train, validation and test
5. Tokenize with the BERT tokenizer
6. Load pre-trained BERT
7. Fine-tune with Hugging Face `Trainer`
8. Evaluate on the test set
9. Predict sentiment on new reviews
10. Save the model
11. Create a Gradio UI

> **Tip:** Run this on a GPU (e.g., Colab → Runtime → Change runtime type → T4 GPU). CPU will work but will be slow.


## 1. Install libraries

In [1]:
!pip install -q transformers datasets scikit-learn accelerate gradio

## 2. Imports & Setup

In [2]:
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import accuracy_score, f1_score

In [3]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


## 3. Load the IMDB Dataset

We load IMDB directly from Hugging Face's `datasets` hub — no Kaggle login needed. It comes pre-split into `train` and `test` (25,000 reviews each).

In [6]:
dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


## 4. Look at a sample

Each example has two fields:
- `text`: the movie review
- `label`: `0` = negative, `1` = positive

In [8]:
sample = dataset["train"][0]
print("Label:", sample["label"], "(0 = negative, 1 = positive)")
print("\nReview:", sample["text"][:500], "...")

Label: 0 (0 = negative, 1 = positive)

Review: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent ...


## 5. Take a small subset (for fast training)

In [9]:
TRAIN_SIZE = 2000
TEST_SIZE = 500

small_train = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
small_test = dataset["test"].shuffle(seed=SEED).select(range(TEST_SIZE))

# split the training data into train + validation
split = small_train.train_test_split(test_size=0.1, seed=SEED)
train_ds = split["train"]
val_ds = split["test"]

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))
print("Test size :", len(small_test))

Train size: 1800
Val size: 200
Test size : 500


## 6. Tokenize the text

In [10]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
  return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

# tokenize all three sets
train_tokenized = train_ds.map(tokenize, batched=True)
val_tokenized = val_ds.map(tokenize, batched=True)
test_tokenized = small_test.map(tokenize, batched=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## 7. Load pre-trained BERT for Classification

`AutoModelForSequenceClassification` loads BERT and adds a small classification head on top (`num_labels=2` for positive/negative). The BERT body starts pre-trained; the classification head starts random. Fine-tuning trains both together on our data.

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 8. Metrics and Data Collator

- **Metrics**: accuracy and F1 score, computed after each epoch.
- **Data collator**: pads each batch to the length of the longest review *in that batch* (more efficient than padding everything to 256).

In [12]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=1)
  return {
      "accuracy": accuracy_score(labels, preds),
      "f1": f1_score(labels, preds)
  }

In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 9. Set up Trainer

In [15]:
training_args = TrainingArguments(
    output_dir="./bert_imdb_output",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 10. Fine-tuning the model

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.357551,0.333488,0.875000,0.873096
2,0.262810,0.503890,0.875000,0.874372


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=450, training_loss=0.34035043080647787, metrics={'train_runtime': 60.9943, 'train_samples_per_second': 59.022, 'train_steps_per_second': 7.378, 'total_flos': 473271010828800.0, 'train_loss': 0.34035043080647787, 'epoch': 2.0})

## 11. Evaluate on test set

In [17]:
final_results = trainer.evaluate(eval_dataset=test_tokenized)
print("Final results:")
for k, v in final_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.262810,0.431947,2,0.878000,0.881553


Final results:
  eval_loss: 0.4319
  eval_accuracy: 0.8780
  eval_f1: 0.8816


## 12. Predict sentiment on new reviews

In [18]:
id2label = {0: "negative", 1: "positive"}

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()

    return {
        "label": id2label[pred_id],
        "confidence": round(probs[pred_id].item(), 4),
    }

# Try it out
reviews = [
    "This movie was absolutely fantastic. Brilliant acting and a gripping story.",
    "I hated this film. It was boring, too long, and badly written.",
    "It was okay — some good moments, but overall forgettable.",
]

for r in reviews:
    print(predict_sentiment(r), "->", r[:60], "...")

{'label': 'positive', 'confidence': 0.9942} -> This movie was absolutely fantastic. Brilliant acting and a  ...
{'label': 'negative', 'confidence': 0.9909} -> I hated this film. It was boring, too long, and badly writte ...
{'label': 'negative', 'confidence': 0.8833} -> It was okay — some good moments, but overall forgettable. ...


## 13. Saving the Model and Tokenizer

In [19]:
SAVE_DIR = "./imdb_bert"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved model and tokenizer to:", SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model and tokenizer to: ./imdb_bert


## 14. Gradio UI

In [20]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ---- Reload from saved directory ----
SAVE_DIR = "./imdb_bert"

reloaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
reloaded_model.to(device)
reloaded_model.eval()

print("Model and tokenizer reloaded on:", device)

# ---- Inference function ----
id2label = {0: "negative", 1: "positive"}
MAX_LENGTH = 256

def predict_sentiment(text):
    inputs = reloaded_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        logits = reloaded_model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()
    return id2label[pred_id], probs[pred_id].item()

# ---- Gradio UI ----
def classify_review(review):
    if not review.strip():
        return "Please enter a review."
    label, confidence = predict_sentiment(review)
    emoji = "😊" if label == "positive" else "😞"
    return f"{emoji} **{label.upper()}**  (confidence: {confidence:.2%})"

demo = gr.Interface(
    fn=classify_review,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Type a movie review here...",
        label="Movie Review"
    ),
    outputs=gr.Markdown(label="Prediction"),
    title="🎬 IMDB Sentiment Classifier",
    description="Fine-tuned BERT predicting whether a movie review is positive or negative.",
    flagging_mode="never",
    examples=[
        ["This movie was absolutely fantastic. Brilliant acting and a gripping story."],
        ["I hated this film. It was boring, too long, and badly written."],
        ["It was okay — some good moments, but overall forgettable."],
    ],
)

demo.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model and tokenizer reloaded on: cuda
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a61979f507277da0ae.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Wrap-up

That's a full BERT sentiment classifier in ~80 lines of code. What we covered:

- Loaded IMDB from Hugging Face `datasets`
- Tokenized reviews with the BERT tokenizer
- Loaded `bert-base-uncased` with a classification head
- Fine-tuned using `Trainer` (handles loop, eval, mixed precision, checkpointing)
- Evaluated with accuracy and F1
- Ran inference on new text
- Saved the trained model
- reloaded the model and created a gradio UI

**Ideas to extend this:**
- Use the full 25k train set and 3–4 epochs — accuracy should exceed 93%
- Try `distilbert-base-uncased` — ~60% the size of BERT, almost as accurate, much faster
- Plot the training loss curve from `trainer.state.log_history`